In [ ]:
# cumulatively sum AFLB for specified target based on priorities
# cfolkers
# WLRS GSS
# 2026 06 29

import geopandas as gpd
import pandas as pd
import numpy as np
stand=gpd.read_parquet(r"path/to/geo/parquet/rvz_rec_for.parquet")
targets=pd.read_csv(r"path/to/csv/river_valley_shortfalls.csv")
targets.rename(columns={"rec_cat_short":"Rec_Cat_sh","WATER_MANAGEMENT_BASIN_NAME":"WATER_MANA"}, inplace=True)
targets["shorfall_area"] = pd.to_numeric(targets["shorfall_area"], errors="coerce")
output=r"path/to/output/rvz_rec_for_balanced.parquet"

In [ ]:

def allocate_shortfall(
    stands_gdf,
    targets_df
) -> gpd.GeoDataFrame:

    stands = stands_gdf.copy()
    targets = targets_df.copy()

    stands["area_ha"] = stands.geometry.area / 10000
    stands["key"] = (
        stands["WATER_MANA"].astype(str).str.strip()
        + "|" +
        stands["Rec_Cat_sh"].astype(str).str.strip()
    )

    targets["key"] = (
        targets["WATER_MANA"].astype(str).str.strip()
        + "|" +
        targets["Rec_Cat_sh"].astype(str).str.strip()
    )


    stands = stands.merge(
        targets[["key", "shorfall_area"]],
        on="key",
        how="left"
    )

    stands["is_rvpz"] = stands["Details"].eq("River Valleys Protection Zone")


    stands["Rec_Cat"] = pd.to_numeric(stands["Rec_Cat"], errors="coerce")
    stands["SIFA_2"] = pd.to_numeric(stands["SIFA_2"], errors="coerce")

    stands = stands.sort_values(
        by=[
            "key",
            "is_rvpz",        
            "Rec_Cat",        
            "SIFA_2",         
            "area_ha"        
        ],
        ascending=[True, False, True, False, False]
    )

    selected_idx = []

    for key, grp in stands.groupby("key", sort=False):

        target = grp["shorfall_area"].iloc[0]
        if pd.isna(target):
            continue

        total = 0.0
        best_total = None
        best_idx = []

        current_idx = []

        for idx, row in grp.iterrows():
            current_idx.append(idx)
            total += row["area_ha"]

            if best_total is None or abs(total - target) < abs(best_total - target):
                best_total = total
                best_idx = current_idx.copy()

            if np.isclose(total, target):
                break

        selected_idx.extend(best_idx)

    return stands.loc[selected_idx].drop(columns=["key", "is_rvpz"])


In [ ]:
results=allocate_shortfall(stand,targets)
results.to_parquet(output)